In [0]:
circuits_df = spark.read.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load("dbfs:/Volumes/formula1/bronze/raw_files/raw/circuits.csv")

display(circuits_df)

In [0]:
#reading and flattening the driver's.json

from pyspark.sql.functions import col

drivers_df = spark.read.format("json") \
.load("dbfs:/Volumes/formula1/bronze/raw_files/raw/drivers.json")

drivers_flat_df = drivers_df.select(
    col("driverId"),
    col("driverRef"),
    col("number"),
    col("code"),
    col("name.forename").alias("forename"),
    col("name.surname").alias("surname"),
    col("dob"),
    col("nationality"),
    col("url")
)

display(drivers_flat_df)

In [0]:
#read constructor.json file 

constructors_df = spark.read.format("json") \
.load("dbfs:/Volumes/formula1/bronze/raw_files/raw/constructors.json")

display(constructors_df)

In [0]:
#read races.csv file 

races_df = spark.read.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load("dbfs:/Volumes/formula1/bronze/raw_files/raw/races.csv")

display(races_df)

In [0]:
#read pitstop.json file 

pitstops_df = spark.read.format("json") \
.option("multiLine","true") \
.load("dbfs:/Volumes/formula1/bronze/raw_files/raw/pit_stops.json")

display(pitstops_df)

In [0]:
#read result.json

results_df = spark.read.format("json") \
.load("dbfs:/Volumes/formula1/bronze/raw_files/raw/results.json")

display(results_df)

In [0]:
#read laptime folders
lap_times_df = spark.read.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load("dbfs:/Volumes/formula1/bronze/raw_files/raw/lap_times/*.csv")

display(lap_times_df)

In [0]:
# checking file type qualifying folder 

display(
    dbutils.fs.ls("dbfs:/Volumes/formula1/bronze/raw_files/raw/qualifying/")
)

In [0]:
#normal json failed to read qualifying folder it creates _corrupt_folder_

qualifying_df = spark.read \
.option("multiline","true") \
.json("dbfs:/Volumes/formula1/bronze/raw_files/raw/qualifying/*.json")

display(qualifying_df)

In [0]:
#schema check 

try:
    qualifying_df
except NameError:
    qualifying_df = spark.read \
    .option("multiline","true") \
    .json("dbfs:/Volumes/formula1/bronze/raw_files/raw/qualifying/*.json")

qualifying_df.printSchema()

In [0]:
display(qualifying_df)

In [0]:
#now we are saving into silver 
try:
    qualifying_df
except NameError:
    qualifying_df = spark.read \
        .option("multiline","true") \
        .json("dbfs:/Volumes/formula1/bronze/raw_files/raw/qualifying/*.json")

spark.sql("CREATE SCHEMA IF NOT EXISTS formula1.silver")
spark.sql("CREATE VOLUME IF NOT EXISTS formula1.silver.qualifying")

qualifying_df.write.format("delta") \
.mode("overwrite") \
.save("/Volumes/formula1/silver/qualifying/")